# Code-aware retrieval

Code-aware retrieval follows relationships written inside source code. It does
not guess file roles from folder names. Python and MATLAB files may be stored
in organized repositories or together in one flat folder.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


## 1. Reuse natural source units

The source extensions from tutorial 50 already identify files, classes,
functions, and methods. The graph reuses those units rather than finding the
same entities again.


In [ ]:
from qachat import scan_repository

report = scan_repository(
    root,
    extensions=[".py", ".m"],
    include_paths=["qachat"],
)

print(f"Selected files: {len(report['selected_files'])}")


In [ ]:
from qachat import (
    extract_source_units,
    read_repository_files,
)

source_files = read_repository_files(report)
units = extract_source_units(source_files)

print(f"Natural units: {len(units)}")


In [ ]:
def count_words(text: str) -> int:
    return len(text.split())


In [ ]:
from qachat import chunk_source_units

chunks = chunk_source_units(
    units,
    count_tokens=count_words,
    target_tokens=300,
    max_tokens=400,
    overlap_tokens=40,
)

print(f"Searchable chunks: {len(chunks)}")


## 2. Build a code graph

The shared graph creates entities and `contains` relationships from natural
units. Small language extensions add calls, imports, and inheritance.


In [ ]:
from qachat import build_code_graph

code_graph = build_code_graph(source_files, units)

print(f"Entities: {len(code_graph.entities)}")
print(f"Relationships: {len(code_graph.relations)}")


In [ ]:
for relation in code_graph.relations:
    print(
        f"{relation.source} "
        f"--{relation.relation}--> "
        f"{relation.target}"
    )


We can also choose one specific entity and inspect its immediate
relationships:

- **Incoming** relationships show which code points to or contains the entity.
- **Outgoing** relationships show which code the entity points to or uses.

Here we inspect `qachat.chat._check_temperature`.

In [ ]:
target = "qachat.chat._check_temperature"
print(target)


In [ ]:
print("Incoming:")
for relation in code_graph.relations:
    if relation.target == target:
        print(
            f"  {relation.source} "
            f"--{relation.relation}--> {target}"
        )


In [ ]:
print("Outgoing:")
for relation in code_graph.relations:
    if relation.source == target:
        print(
            f"  {target} "
            f"--{relation.relation}--> {relation.target}"
        )


## 3. Resolve an explicit symbol

This step is deterministic. The question must name a code entity; no LLM
guesses which entity exists.


In [ ]:
from qachat import resolve_code_entities

QUESTION = "Which code is related to semantic_search?"
matches = resolve_code_entities(code_graph, QUESTION)

for match in matches:
    print(f"{match.kind}: {match.name}")


## 4. Retrieve its code relationships

Distance 0 is the named entity. Distance 1 is directly connected through a
call, import, inheritance, or containment relationship.


In [ ]:
from qachat import code_search

results = code_search(
    code_graph,
    chunks,
    QUESTION,
    count=5,
    max_distance=1,
)

for rank, result in enumerate(results, start=1):
    print(
        f"{rank}. distance={result.distance} "
        f"symbol={result.chunk.symbol}"
    )


In [ ]:
for result in results:
    print(f"\n>>> {result.entity} <<<\n")
    print(result.chunk.text)


## Result

QAChat reused natural source units, added Python-specific or MATLAB-specific
relationships, and retrieved code connected to an explicitly named symbol.
